In [0]:
# ============================================
# GOLD LAYER — Advanced feature engineering
# ============================================

from pyspark.sql.functions import (
    col, avg, stddev, count,
    sum as spark_sum,
    max as spark_max,
    min as spark_min,
    round as spark_round,
    current_timestamp,
    percentile_approx,
    when
)
from pyspark.sql.window import Window

# ── 1. Load Silver ───────────────────────────────────────────────
df_silver = spark.table("fraud_db.silver_transactions")
print(f"✅ Silver rows loaded: {df_silver.count()}")

# ── 2. Hourly aggregation summary ────────────────────────────────
df_hourly = df_silver.groupBy("hour_of_day").agg(
    count("*")                         .alias("txn_count"),
    spark_sum("Class")                 .alias("fraud_count"),
    avg("Amount")                      .alias("avg_amount"),
    stddev("Amount")                   .alias("std_amount"),
    spark_max("Amount")                .alias("max_amount"),
    spark_min("Amount")                .alias("min_amount"),
    spark_sum("is_high_value")         .alias("high_value_count"),
    percentile_approx("Amount", 0.95)  .alias("p95_amount")
)

df_hourly = df_hourly.withColumn(
    "fraud_rate",
    spark_round(
        col("fraud_count") / col("txn_count") * 100,
        4
    )
)

print("\n=== HOURLY FRAUD PATTERNS ===")
df_hourly.orderBy("hour_of_day").show(24)

# ── 3. Window features on transactions ───────────────────────────
window_spec = Window.orderBy("Time").rowsBetween(-100, 0)

df_gold = df_silver \
    .withColumn(
        "rolling_avg_amount",
        spark_round(avg("Amount").over(window_spec), 2)
    ) \
    .withColumn(
        "rolling_txn_count",
        count("*").over(window_spec)
    ) \
    .withColumn(
        "rolling_fraud_rate",
        spark_round(
            spark_sum("Class").over(window_spec) /
            count("*").over(window_spec) * 100,
            4
        )
    ) \
    .withColumn(
        "amount_vs_avg_ratio",
        spark_round(
            col("Amount") / col("rolling_avg_amount"),
            4
        )
    ) \
    .withColumn(
        "is_amount_spike",
        when(col("amount_vs_avg_ratio") > 3, 1).otherwise(0)
    ) \
    .withColumn("gold_timestamp", current_timestamp())

# ── 4. Preview gold features ─────────────────────────────────────
print("\n=== GOLD FEATURES PREVIEW ===")
df_gold.select(
    "Time", "Amount", "hour_of_day",
    "is_night", "rolling_avg_amount",
    "amount_vs_avg_ratio", "is_amount_spike",
    "rolling_fraud_rate", "Class"
).show(10)

# ── 5. Feature correlation with fraud ────────────────────────────
print("\n=== FEATURE CORRELATIONS WITH FRAUD ===")
feature_cols = [
    "Amount", "is_high_value", "is_night",
    "amount_vs_avg_ratio", "is_amount_spike",
    "rolling_fraud_rate", "hour_of_day"
]
for feat in feature_cols:
    corr = df_gold.stat.corr(feat, "Class")
    bar  = "█" * int(abs(corr) * 50)
    print(f"  {feat:25s} {corr:+.6f}  {bar}")

# ── 6. Top fraud risk hours ───────────────────────────────────────
print("\n=== TOP 5 HIGHEST FRAUD RATE HOURS ===")
df_hourly.orderBy(col("fraud_rate").desc()).show(5)

# ── 7. Write Gold Delta tables ───────────────────────────────────
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.gold_features")

df_hourly.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_db.gold_hourly_summary")

print(f"\n✅ Gold features saved!")
print(f"   Rows written  : {df_gold.count()}")
print(f"   Table 1       : fraud_db.gold_features")
print(f"   Table 2       : fraud_db.gold_hourly_summary")



✅ Silver rows loaded: 281918

=== HOURLY FRAUD PATTERNS ===
+-----------+---------+-----------+------------------+------------------+----------+----------+----------------+----------+----------+
|hour_of_day|txn_count|fraud_count|        avg_amount|        std_amount|max_amount|min_amount|high_value_count|p95_amount|fraud_rate|
+-----------+---------+-----------+------------------+------------------+----------+----------+----------------+----------+----------+
|          0|     7578|          5| 61.14368962786969| 185.0825034880527|   7712.43|      0.01|              38|     255.2|     0.066|
|          1|     4177|         10| 63.23343069188421| 146.4452763851125|    2481.6|      0.01|              20|    261.22|    0.2394|
|          2|     3286|         48| 70.87825623858814|380.74431765708704|   18910.0|      0.01|              24|     249.6|    1.4607|
|          3|     3453|         16| 52.27126556617454|136.80603401397582|    2793.6|      0.01|              16|     197.0|    0.4

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+------+-----------+--------+------------------+-------------------+---------------+------------------+-----+
|Time|Amount|hour_of_day|is_night|rolling_avg_amount|amount_vs_avg_ratio|is_amount_spike|rolling_fraud_rate|Class|
+----+------+-----------+--------+------------------+-------------------+---------------+------------------+-----+
| 0.0|  2.69|          0|       1|              2.69|                1.0|              0|               0.0|    0|
| 0.0|149.62|          0|       1|             76.16|             1.9645|              0|               0.0|    0|
| 1.0|378.66|          0|       1|            176.99|             2.1394|              0|               0.0|    0|
| 1.0| 123.5|          0|       1|            163.62|             0.7548|              0|               0.0|    0|
| 2.0|  3.67|          0|       1|            131.63|             0.0279|              0|               0.0|    0|
| 2.0| 69.99|          0|       1|            121.36|             0.5767|       